# 1.3 — Visualização das features (Forma 1, InfluxDB)

**Forma 1 — Edge / janela fixa.** O ESP32 (`app17-9`) calculou as 7 features por janela e o
fluxo Node-RED gravou no measurement `vibracao_features` do **InfluxDB Cloud**.
Aqui nós lemos essas features e comparamos visualmente **normal × anomalia**.

> **Atende:** Sprint 3 – item 4 (visualização) e prepara a Sprint 4 (base analítica).
> Lembre que na Forma 1 **não temos o sinal bruto** — só o resumo da janela.

Roda no **Google Colab** (a nuvem é alcançável de qualquer lugar; o `localhost` não seria).

In [ ]:
!pip install -q influxdb-client pandas matplotlib

## Conexão com o InfluxDB Cloud

Preencha com os **seus** valores (os mesmos do nó InfluxDB do Node-RED).

In [1]:
from influxdb_client import InfluxDBClient
import pandas as pd
import matplotlib.pyplot as plt

INFLUX_URL    = "https://us-east-1-1.aws.cloud2.influxdata.com"  # URL da sua regiao
INFLUX_TOKEN  = "KXTPf0peaYQU-QMGu-yJNWwVbBLNoUMmNwBBsrfcnK5GseDHLs_QZx7hNW4sToLnp1qeEXu5CwUq6rwf30FcXQ=="                          # token de leitura
INFLUX_ORG    = "e044ac59f07be199"                                         # nome ou ID da org
INFLUX_BUCKET = "IoTSensores"                                        # bucket de destino
MEASUREMENT   = "vibracao_features"

client = InfluxDBClient(url=INFLUX_URL, token=INFLUX_TOKEN, org=INFLUX_ORG)

ModuleNotFoundError: No module named 'influxdb_client'

## Consulta (Flux) → DataFrame

O `pivot` transforma cada field (mean_ax, std_ax, ...) em coluna; cada linha = uma janela.

In [ ]:
flux = f'''
from(bucket: "{INFLUX_BUCKET}")
  |> range(start: -30d)
  |> filter(fn: (r) => r._measurement == "{MEASUREMENT}")
  |> pivot(rowKey: ["_time"], columnKey: ["_field"], valueColumn: "_value")
  |> keep(columns: ["_time", "label",
        "mean_ax", "mean_ay", "mean_az",
        "std_ax", "std_ay", "std_az", "rms_mag"])
  |> sort(columns: ["_time"])
'''

df = client.query_api().query_data_frame(flux)
if isinstance(df, list):
    df = pd.concat(df, ignore_index=True)

df = df.rename(columns={"_time": "time"})
df = df[["time", "label", "mean_ax", "mean_ay", "mean_az",
         "std_ax", "std_ay", "std_az", "rms_mag"]]
print("Total de janelas:", len(df))
print(df["label"].value_counts())
df.head()

## Série temporal das features por classe

Esperamos que `std_*` e `rms_mag` **subam** na anomalia (mesa/ESP32 chacoalhando) e fiquem
baixos no normal (parado). A média (`mean_*`) muda pouco, pois vibração positiva e negativa
se cancela (Aula 14, slide 12).

In [ ]:
cores = {"ligado_normal": "tab:green", "ligado_anomalia": "tab:red"}

fig, axes = plt.subplots(2, 1, figsize=(11, 7), sharex=True)
for lab, g in df.groupby("label"):
    c = cores.get(lab, "tab:blue")
    axes[0].plot(g["time"], g["rms_mag"], ".", label=lab, color=c)
    axes[1].plot(g["time"], g["std_ax"], ".", label=lab, color=c)
axes[0].set_ylabel("rms_mag (m/s²)"); axes[0].legend(); axes[0].set_title("Intensidade por janela")
axes[1].set_ylabel("std_ax (m/s²)"); axes[1].legend(); axes[1].set_xlabel("tempo")
plt.tight_layout(); plt.show()

## Comparação direta das classes (boxplots)

Boxplot de cada feature separando as duas classes — quanto menos as caixas se sobrepõem,
mais fácil o modelo separa normal de anomalia.

In [ ]:
FEATURES = ["mean_ax", "mean_ay", "mean_az", "std_ax", "std_ay", "std_az", "rms_mag"]

fig, axes = plt.subplots(2, 4, figsize=(15, 7))
for ax, feat in zip(axes.ravel(), FEATURES):
    dados = [df[df["label"] == lab][feat].dropna() for lab in df["label"].unique()]
    ax.boxplot(dados, labels=list(df["label"].unique()))
    ax.set_title(feat); ax.tick_params(axis="x", rotation=20)
axes.ravel()[-1].axis("off")
plt.tight_layout(); plt.show()

## Conclusão

- Se `rms_mag` / `std_*` separam bem as classes, há sinal suficiente para o classificador
  (notebook **1.5**).
- **Limitação da Forma 1:** não dá para inspecionar o sinal bruto nem testar outra janela —
  só as features chegaram ao banco. Quem guarda o raw é a **Forma 2** (notebooks 2.x).